# SafeScan - Step 1 (v2): IIT Patna Indian Grocery Dataset - Download, Crop, Categorize, OCR

This replaces the Freiburg-based Step 1. This notebook:
1. Downloads the Grocery_Items dataset (IIT Patna, via Roboflow) - 6,695 images, object detection format
2. Crops individual products out of each image using the bounding box annotations
3. Collapses the 4,400+ specific brand classes into ~20 broad food categories (keyword-based)
4. Runs OCR on every cropped product image
5. Saves everything in the same folder structure used by Step 2 (category -> images + JSON)

**Runtime setup:** Runtime -> Change runtime type -> GPU (T4)


## Cell 1 - Install packages

In [1]:
!pip install roboflow easyocr -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 127.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 27.7 MB/s eta 0:00:00


## Cell 2 - Download the dataset from Roboflow

Paste your Roboflow API key below (Settings -> API Keys on roboflow.com).
Check the "Dataset" tab on the project page for the correct version number - update
`VERSION_NUMBER` if it's not 1.


In [2]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "NF0VRF6Z7nqBqFXUHEeG"
VERSION_NUMBER = 45   # check the "Dataset" tab on the project page and update if needed

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("iit-patna-qg1jh").project("grocery_items-7i2em")
version = project.version(VERSION_NUMBER)
dataset = version.download("coco")  # COCO format gives us images + bounding box annotations

print("Downloaded to:", dataset.location)


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Grocery_Items-45 in coco:: 100%|██████████| 5991/5991 [00:20<00:00, 287.53it/s]


Downloaded to: /content/Grocery_Items-45


In [3]:
import os
print(os.listdir(dataset.location))


['README.roboflow.txt', 'README.dataset.txt', 'train', 'test', 'valid']


**STOP HERE.** Check the printed output. COCO format usually gives folders like
`train/`, `valid/`, `test/`, each containing images plus an `_annotations.coco.json` file.
If the folder names differ, update `SPLIT_FOLDERS` in Cell 4 below to match.


In [4]:
train_path = os.path.join(dataset.location, "train")
print(os.listdir(train_path)[:10])  # should show image files + _annotations.coco.json

['IMG_20230314_184025_jpg.rf.281fadd62255b42185baed9790a3af6c.jpg', 'image_224_jpg.rf.892a290bd44937cbce42986091c5b688.jpg', '20230812_083626_jpg.rf.20bc8011bc2fd3eb482526f84eccd959.jpg', 'IMG20240107192550_BURST005_jpg.rf.c32e48b330ea9215cc5ff622cecac128.jpg', 'image_375_jpg.rf.15df3b0597f6272b6ce4e667874bb04f.jpg', '_storage_emulated_0_DCIM_-convert_tmp_files_IMG20240107191636_20240108113522_jpg.rf.0e30e4463221dfba9fd2211c672ce791.jpg', '1695021836322_jpg.rf.716ba804b52e0bc565644baf6ac2347a.jpg', 'image_843_jpg.rf.52036661476538a770ef9db8fe091ea9.jpg', 'image_19_jpg.rf.0274b2ce73af31c30cd03196e706ea51.jpg', 'IMG-20230821-WA0082_jpg.rf.2a692ab978712c3af5aa65b112bfd5bd.jpg']


## Cell 3 - Define the category collapsing map (4,400+ classes -> ~20 broad food categories)

This uses keyword matching against each original class name. Classes that don't match any
food-related keyword (e.g. toothpaste, air freshener, detergent) are dropped, since they are
not relevant to nutrition/allergen analysis. You can edit these keyword lists any time to
improve the mapping.


In [12]:
CATEGORY_KEYWORDS = {
    "TEA":            ["tea"],
    "MILK":           ["milk"],
    "BUTTER":         ["butter"],
    "CHEESE":         ["cheese"],
    "RICE":           ["rice"],
    "SUGAR":          ["sugar"],
    "SALT":           ["salt"],
    "SPICES":         ["spice", "masala", "chilli", "turmeric", "pepper", "cumin", "coriander", "powder"],
    "BISCUITS":       ["biscuit", "cookie"],
    "CHOCOLATE_CANDY":["chocolate", "candy", "choco", "sweet"],
    "CHIPS_SNACKS":   ["chips", "fryums", "snack", "namkeen", "bhujia", "popcorn"],
    "SAUCE_KETCHUP":  ["sauce", "ketchup", "chutney", "vinegar"],
    "OIL":            ["oil"],
    "CEREAL":         ["cereal", "cornflakes", "oats", "muesli"],
    "SOFTDRINK_JUICE":["juice", "softdrink", "cola", "soda", "drink", "fizz"],
    "PICKLE":         ["pickle", "achar"],
    "HONEY":          ["honey"],
    "NOODLES_PASTA":  ["noodle", "pasta", "maggi", "vermicelli"],
    "PULSES_DAL":     ["dal", "pulse", "lentil", "chana", "rajma"],
    "WATER":          ["water"],
    "ALMONDS_NUTS":   ["almond", "nut", "cashew", "pista"],
}

def map_to_broad_category(class_name):
    """Returns the broad category for a given original class name, or None if no match."""
    name_lower = class_name.lower()
    for broad_cat, keywords in CATEGORY_KEYWORDS.items():
        for kw in keywords:
            if kw in name_lower:
                return broad_cat
    return None

# quick test
test_names = ["Aashirvaad_ChilliPowder", "Almondo_Amul", "ToothPaste_Colgate", "Butter_Amul"]
for n in test_names:
    print(f"{n} -> {map_to_broad_category(n)}")


Aashirvaad_ChilliPowder -> SPICES
Almondo_Amul -> ALMONDS_NUTS
ToothPaste_Colgate -> None
Butter_Amul -> BUTTER


## Cell 4 - Crop products from each image using bounding boxes, sorted into broad categories

In [13]:
import json
from PIL import Image

SPLIT_FOLDERS = ["train", "valid", "test"]   # update based on Cell 2's printed output
CROPPED_OUTPUT_ROOT = "/content/iitpatna_cropped"

os.makedirs(CROPPED_OUTPUT_ROOT, exist_ok=True)

def crop_and_categorize(dataset_location, split_folders, output_root, max_per_class=None):
    category_counts = {}

    for split in split_folders:
        split_path = os.path.join(dataset_location, split)
        ann_path = os.path.join(split_path, "_annotations.coco.json")
        if not os.path.exists(ann_path):
            print(f"Skipping {split} - no annotations file found")
            continue

        with open(ann_path, 'r') as f:
            coco = json.load(f)

        # Build lookup: category_id -> original class name
        cat_id_to_name = {c['id']: c['name'] for c in coco['categories']}
        # Build lookup: image_id -> file_name
        img_id_to_file = {img['id']: img['file_name'] for img in coco['images']}

        print(f"\nProcessing split: {split} ({len(coco['annotations'])} annotations)")

        for ann in coco['annotations']:
            orig_class_name = cat_id_to_name.get(ann['category_id'], "")
            broad_cat = map_to_broad_category(orig_class_name)
            if broad_cat is None:
                continue  # not a food-relevant class, skip

            if max_per_class and category_counts.get(broad_cat, 0) >= max_per_class:
                continue

            img_file = img_id_to_file.get(ann['image_id'])
            if img_file is None:
                continue
            img_path = os.path.join(split_path, img_file)
            if not os.path.exists(img_path):
                continue

            try:
                img = Image.open(img_path).convert("RGB")
                x, y, w, h = ann['bbox']  # COCO format: [x_min, y_min, width, height]
                cropped = img.crop((x, y, x + w, y + h))

                cat_folder = os.path.join(output_root, broad_cat)
                os.makedirs(cat_folder, exist_ok=True)

                count = category_counts.get(broad_cat, 0)
                save_path = os.path.join(cat_folder, f"{broad_cat}_{count:04d}.jpg")
                cropped.save(save_path)

                category_counts[broad_cat] = count + 1
            except Exception as e:
                print(f"  Error cropping annotation {ann['id']}: {e}")

    return category_counts

# NOTE: set max_per_class=20 first for a quick pipeline test, then re-run with None for full data
category_counts = crop_and_categorize(dataset.location, SPLIT_FOLDERS, CROPPED_OUTPUT_ROOT, max_per_class=800)
print("\nFinal category counts:", category_counts)



Processing split: train (62747 annotations)

Processing split: valid (9462 annotations)

Processing split: test (4097 annotations)

Final category counts: {'BISCUITS': 425, 'PULSES_DAL': 800, 'NOODLES_PASTA': 800, 'CHEESE': 800, 'MILK': 800, 'ALMONDS_NUTS': 684, 'CHOCOLATE_CANDY': 582, 'CHIPS_SNACKS': 303, 'OIL': 800, 'SOFTDRINK_JUICE': 800, 'SAUCE_KETCHUP': 310, 'BUTTER': 800, 'RICE': 717, 'SPICES': 800, 'CEREAL': 248, 'SUGAR': 444, 'SALT': 178, 'PICKLE': 752, 'HONEY': 160, 'TEA': 497, 'WATER': 236}


**Note on the quick test run:** same idea as before - `max_per_class=20` lets you verify
cropping and categorization work correctly in a few minutes. Once confirmed, change it to
`max_per_class=None` and re-run Cell 4 for the full dataset.


## Cell 5 - Set up OCR reader

In [14]:
import easyocr
reader = easyocr.Reader(['en'])
print("EasyOCR reader ready.")


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteEasyOCR reader ready.


## Cell 6 - Run OCR on the cropped, categorized images

In [15]:
OUTPUT_TEXT_ROOT = "/content/iitpatna_ocr_text"
os.makedirs(OUTPUT_TEXT_ROOT, exist_ok=True)

def run_ocr_on_dataset(images_root, output_root):
    categories = sorted(os.listdir(images_root))
    print(f"Found {len(categories)} categories: {categories}")

    for category in categories:
        cat_img_path = os.path.join(images_root, category)
        if not os.path.isdir(cat_img_path):
            continue

        cat_output_path = os.path.join(output_root, category)
        os.makedirs(cat_output_path, exist_ok=True)

        img_files = [f for f in os.listdir(cat_img_path) if f.lower().endswith('.jpg')]
        print(f"\nProcessing category: {category} ({len(img_files)} images)")

        for idx, img_file in enumerate(img_files):
            img_path = os.path.join(cat_img_path, img_file)
            try:
                results = reader.readtext(img_path)
                ocr_json = [{"description": text, "confidence": float(conf)}
                            for (bbox, text, conf) in results]
                json_filename = img_file.rsplit('.', 1)[0] + '.json'
                json_path = os.path.join(cat_output_path, json_filename)
                with open(json_path, 'w', encoding='utf-8') as f:
                    json.dump(ocr_json, f, ensure_ascii=False, indent=2)
            except Exception as e:
                print(f"  Error processing {img_file}: {e}")

            if (idx + 1) % 20 == 0:
                print(f"  Processed {idx + 1}/{len(img_files)} images...")

    print("\nOCR extraction complete!")

run_ocr_on_dataset(CROPPED_OUTPUT_ROOT, OUTPUT_TEXT_ROOT)


Found 21 categories: ['ALMONDS_NUTS', 'BISCUITS', 'BUTTER', 'CEREAL', 'CHEESE', 'CHIPS_SNACKS', 'CHOCOLATE_CANDY', 'HONEY', 'MILK', 'NOODLES_PASTA', 'OIL', 'PICKLE', 'PULSES_DAL', 'RICE', 'SALT', 'SAUCE_KETCHUP', 'SOFTDRINK_JUICE', 'SPICES', 'SUGAR', 'TEA', 'WATER']

Processing category: ALMONDS_NUTS (684 images)
  Processed 20/684 images...
  Processed 40/684 images...
  Processed 60/684 images...
  Processed 80/684 images...
  Processed 100/684 images...
  Processed 120/684 images...
  Processed 140/684 images...
  Processed 160/684 images...
  Processed 180/684 images...
  Processed 200/684 images...
  Processed 220/684 images...
  Processed 240/684 images...
  Processed 260/684 images...
  Processed 280/684 images...
  Processed 300/684 images...
  Processed 320/684 images...
  Processed 340/684 images...
  Processed 360/684 images...
  Processed 380/684 images...
  Processed 400/684 images...
  Processed 420/684 images...
  Processed 440/684 images...
  Processed 460/684 images...

## Cell 7 - Sanity check + download everything

In [16]:
sample_category = os.listdir(OUTPUT_TEXT_ROOT)[0]
sample_folder = os.path.join(OUTPUT_TEXT_ROOT, sample_category)
sample_json = os.listdir(sample_folder)[0]
with open(os.path.join(sample_folder, sample_json), 'r') as f:
    print(json.load(f))


[]


In [17]:
import os, json

categories = os.listdir(OUTPUT_TEXT_ROOT)
print(f"Checking {len(categories)} categories...\n")

for cat in categories[:8]:
    cat_folder = os.path.join(OUTPUT_TEXT_ROOT, cat)
    json_files = os.listdir(cat_folder)

    found_text = False
    for jf in json_files[:5]:
        with open(os.path.join(cat_folder, jf), 'r') as f:
            data = json.load(f)
        if len(data) > 0:
            print(f"[{cat}] {jf}: {[d['description'] for d in data]}")
            found_text = True
            break
    if not found_text:
        print(f"[{cat}] — no text found in first 5 samples checked")
    print()

Checking 21 categories...

[PICKLE] — no text found in first 5 samples checked

[OIL] OIL_0712.json: ['Cod ', 'OIL', 'CApsules', 'SFACOD', 'Vtt', 'Liver', '~amin', 'MpaReS']

[CHIPS_SNACKS] CHIPS_SNACKS_0215.json: ['"Zp lock', 'Jabsons', 'in', 'Traditions', 'Rich !', 'Mini', 'Bhakharwadi', 'famous', 'Maharashtras', "'Snack", 'Spicy ', 'Sweet ']

[SUGAR] SUGAR_0063.json: ['{0', 'ewnib']

[CEREAL] CEREAL_0174.json: ['Yum', 'Tan', 'KOdte', 'Katoinina']

[CHOCOLATE_CANDY] CHOCOLATE_CANDY_0248.json: ['(NDYMAn', '"Cozt', '"8clains', 'Ton']

[SOFTDRINK_JUICE] SOFTDRINK_JUICE_0167.json: ['E']

[BUTTER] BUTTER_0315.json: ['JUC']



In [18]:
# Zip and download both the cropped images and the OCR text - you'll need both for Step 2
!zip -rq iitpatna_cropped.zip /content/iitpatna_cropped
!zip -rq iitpatna_ocr_text.zip /content/iitpatna_ocr_text

from google.colab import files
files.download('iitpatna_cropped.zip')
files.download('iitpatna_ocr_text.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next step

Once this runs successfully on the full dataset, you're ready for an updated **Step 2**
(feature extraction + training) - same structure as before, just pointing at
`iitpatna_cropped` and `iitpatna_ocr_text` instead of the Freiburg folders. Message Claude
to get that updated notebook.
